In [2]:
import ydf
import pandas as pd
import numpy as np
# run with cleaned dataset and without changing frequency of non IC

In [3]:
crime = pd.read_csv("Crime_Data_from_2020_to_2024.csv")
crime.columns = crime.columns.str.replace(' ', '_')


In [4]:
# Drop redundant/unnecessary columns
crime= crime.drop(columns=["DR_NO", "Date_Rptd", "AREA","Part_1-2","Crm_Cd", 
                   "Mocodes","Premis_Desc","Weapon_Desc","Status_Desc", 
                   "Crm_Cd_4","LOCATION","Cross_Street","LAT","LON"])
crime = crime.drop(columns=['Crm_Cd_1', 'Crm_Cd_2', 'Crm_Cd_3'])
#crime.columns

In [11]:
# Remove Missing Labels + CC
crime = crime.dropna(subset=['Status'])
crime = crime[crime['Status'] != 'CC']

In [6]:
# Set missing values to unknown (X)
crime['Vict_Sex'] = crime['Vict_Sex'].replace(np.nan, "X")
crime['Vict_Sex'] = crime['Vict_Sex'].replace("-", "X")
crime['Vict_Sex'] = crime['Vict_Sex'].replace("H", "X")

crime['Vict_Descent'] = crime['Vict_Descent'].replace(np.nan, "X")
crime['Vict_Descent'] = crime['Vict_Descent'].replace("-", "X")


In [7]:
# Remove negative ages
crime.loc[crime['Vict_Age'] < 0, 'Vict_Age'] = np.nan


In [8]:
def split_dataset(ds, test_ratio):
  test_indices = np.random.rand(len(ds)) < test_ratio
  return ds[~test_indices], ds[test_indices]

crime_train, crime_test = split_dataset(crime, 0.10)
#print(len(crime_train), len(crime_test))


In [8]:
model = ydf.RandomForestLearner(label = "Status", num_trees = 20, missing_value_policy="GLOBAL_IMPUTATION").train(crime_train)

Train model on 904700 examples
Model trained in 0:03:23.769143


In [10]:
model.evaluate(crime_test)

Label \ Pred,IC,AO,AA,JA,JO,CC
IC,77559,6647,6018,266,156,0
AO,2189,4071,1793,37,33,0
AA,298,320,770,23,5,0
JA,1,0,0,4,0,0
JO,0,0,0,1,2,0
CC,0,0,0,0,0,0


In [12]:
model.analyze(crime_test)